# Phase 3 Rhetorical Roles - Model 7 only (full / LoRA / LoRMA)
Sentence-level rhetorical-role classification using **only your Model 7** - the 12-layer BERT-style stack (768 hidden, 12 heads, FFN 3,072, post-LayerNorm, GELU, learned absolute positions, 512 positions, uncased 30,522-entry WordPiece vocabulary) pre-trained from random initialization with the prefix-LM continuation objective and 3-replica DiLoCo. There is no InLegalBERT, no second model and no blending in this notebook.

Model 7 is adapted with the method you pick via `FUSE_METHOD` in the config cell: `"full"` (all weights trained), `"lora"` (frozen base + additive low-rank `A/B`), `"lorma"` (frozen base + multiplicative low-rank update) or `"kd"` (knowledge distillation: the fully fine-tuned model is the teacher, a half-depth student initialised from its pre-trained layers learns from hard labels + the teacher's soft labels; the student makes the submission). At fine-tune time the whole input is treated as the bidirectional *prefix* (no causal suffix mask, segment id 0 everywhere), which is what the prefix part of Model 7 was trained with. The prediction (LM) head and the pooler are not used; a new classification head is added.

Input per sentence: the target sentence plus a window of neighbouring sentences, a domain tag and a relative-position bucket. The classifier produces a probability over the 13 labels per sentence and the argmax is the predicted label. Model selection (early stopping) uses the mean of CL and IT dev macro-F1. Dev scores are printed and saved in `model7_report.json`; `model7_logits.npz` keeps the raw dev/test logits.

Output: `/kaggle/working/submission_rr_model7_<method>.zip` containing `prediction.json` (`{doc_id: [label, ...]}`).

**Attach as Kaggle inputs:** the Model 7 checkpoint dataset (a `.pt` file, default name `best.pt`) and the Model 7 tokenizer dataset (`vocab.txt`, or `tokenizer.json`). No internet is needed except for downloading the competition data from the Hugging Face dataset repo.

In [ ]:
import os, sys, json, math, random, time, zipfile, glob, re, gc
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as grad_checkpoint
from tqdm.auto import tqdm
from huggingface_hub import login, hf_hub_download
from kaggle_secrets import UserSecretsClient
from sklearn.metrics import precision_recall_fscore_support, classification_report

HF_TOKEN = UserSecretsClient().get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
print("Logged in.")

## Config

In [ ]:
# ===================== what to run =====================
RETRAIN_LORA   = True            # True: train Model 7 on the task; False: load LORA_CKPT
LORA_CKPT      = None            # a best_<method>_rr.pt saved by an earlier run of this notebook (used when RETRAIN_LORA=False)

# ===================== data / context =====================
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
P3 = "Phase_3_RR"
PREV_K, NEXT_K = 3, 1            # neighbouring sentences given as context
POS_BUCKETS    = 20              # relative position of the sentence in its document
USE_DOMAIN_TAG = True
CLASS_WEIGHT_POWER = 0.5         # 0 = plain CE, 0.5 = inverse sqrt frequency, 1 = inverse frequency

# ===================== Model 7 =====================
FUSE_METHOD    = "lorma"          # <<< CHOOSE: "full" | "lora" | "lorma" | "kd"  (how Model 7 is adapted for classification)
RANK, ALPHA    = 8, 16.0
A_CTX_TOKENS   = 48              # each context sentence is clipped to this many WordPiece tokens
A_TGT_TOKENS   = 200             # target sentence clip
A_MAX_LEN      = 384             # must be <= 512 (Model 7 position table)
A_INFER_BATCH  = 64
A_PATIENCE     = 3
A_HEAD_LR, A_WD, A_WARMUP = 5e-4, 0.01, 0.06
# per-method presets: backbone LR, epochs, batch, gradient checkpointing (override below if you like)
_PRESETS = {
    "lora":  dict(lr=3e-4, epochs=8, batch=16, gc=False),
    "lorma": dict(lr=3e-4, epochs=8, batch=16, gc=False),
    "full":  dict(lr=5e-5, epochs=8, batch=16, gc=True),
    "kd":    dict(lr=5e-5, epochs=8, batch=16, gc=True),    # student: all weights train; the teacher uses the "full" preset    # every weight trains: lower LR, checkpointing to fit the T4
}
assert FUSE_METHOD in _PRESETS, "FUSE_METHOD must be 'full', 'lora', 'lorma' or 'kd'"
METHOD = FUSE_METHOD
A_LR, A_EPOCHS, A_BATCH, USE_GC = (_PRESETS[FUSE_METHOD][k] for k in ("lr", "epochs", "batch", "gc"))
print(f"Model A method: {FUSE_METHOD} | backbone LR {A_LR} | epochs {A_EPOCHS} | batch {A_BATCH} | grad-checkpointing {USE_GC}")

SEED  = 42
SMOKE = False                    # True = tiny run to check the whole pipeline end to end

# ----------------------------- architecture of Model 7 (BERT-base-style stack) --
M7_CFG = dict(vocab_size=30522, hidden_size=768, num_hidden_layers=12, num_attention_heads=12,
              intermediate_size=3072, hidden_act="gelu", max_position_embeddings=512, type_vocab_size=2,
              layer_norm_eps=1e-12, hidden_dropout_prob=0.1, attention_probs_dropout_prob=0.1,
              position_embedding_type="absolute")
D_MODEL, MAX_SEQ_LEN = M7_CFG["hidden_size"], M7_CFG["max_position_embeddings"]
assert A_MAX_LEN <= MAX_SEQ_LEN
CKPT_KEY_PREFIX = ""             # optional extra prefix to strip from checkpoint keys (e.g. "model."), after "_orig_mod."/"module."/"bert."

# ===================== knowledge distillation (FUSE_METHOD = "kd") =====================
# stage 1: the full-depth model is fully fine-tuned as the TEACHER.
# stage 2: a half-depth STUDENT (initialised from evenly spaced pre-trained layers) is trained on
#          (1-KD_ALPHA) * class-weighted CE(hard labels) + KD_ALPHA * T^2 * KL(teacher soft labels || student). The student makes the submission.
KD_TEACHER_CKPT   = None         # best_teacher_rr.pt from an earlier run -> skips teacher training
KD_TEACHER_LR     = 5e-5
KD_TEMP           = 2.0
KD_ALPHA          = 0.5
KD_STUDENT_LAYERS = M7_CFG["num_hidden_layers"] // 2
KD_LAYER_MAP      = [int(round(x)) for x in np.linspace(0, M7_CFG["num_hidden_layers"] - 1, KD_STUDENT_LAYERS)]
assert len(set(KD_LAYER_MAP)) == KD_STUDENT_LAYERS

CKPT_PATH = None
TOK_DIR   = None
OUT_DIR = "/kaggle/working/rr_model7"
os.makedirs(OUT_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = dev.type == "cuda"
print("device:", dev, "|", torch.cuda.get_device_name(0) if use_amp else "")

## Locate Model 7 checkpoint & tokenizer

In [ ]:
from transformers import BertTokenizerFast, AutoTokenizer


def _find(patterns, prefer=()):
    hits = sorted({h for p in patterns for h in glob.glob(p, recursive=True)})
    for key in prefer:
        pref = [h for h in hits if key in h.lower()]
        if pref:
            return pref[0], hits
    return (hits[0] if hits else None), hits


if CKPT_PATH is None:
    CKPT_PATH, hits = _find(["/kaggle/input/**/best.pt"], prefer=("m7", "model7", "model_7", "capstone"))
    print("best.pt candidates:", hits)
if TOK_DIR is None:
    tj, hits = _find(["/kaggle/input/**/vocab.txt", "/kaggle/input/**/tokenizer.json"], prefer=("m7", "model7", "legal", "tokenizer"))
    TOK_DIR = os.path.dirname(tj) if tj else None
    print("tokenizer candidates:", hits)
assert CKPT_PATH and os.path.isfile(CKPT_PATH), "Model 7 checkpoint not found - attach the checkpoint dataset or set CKPT_PATH"
assert TOK_DIR and os.path.isdir(TOK_DIR), "tokenizer not found - attach the tokenizer dataset or set TOK_DIR"
print("using checkpoint:", CKPT_PATH)
print("using tokenizer :", TOK_DIR)

try:                                      # uncased WordPiece, 30,522 entries
    tok_L = BertTokenizerFast.from_pretrained(TOK_DIR, do_lower_case=True)
except Exception as e:
    print("BertTokenizerFast failed (", e, ") - falling back to AutoTokenizer")
    tok_L = AutoTokenizer.from_pretrained(TOK_DIR)
tok_L.model_max_length = 10 ** 9          # we window ourselves; silence the length warning
VOCAB_SIZE = len(tok_L)
print(f"VOCAB_SIZE={VOCAB_SIZE} CLS={tok_L.cls_token_id} SEP={tok_L.sep_token_id} PAD={tok_L.pad_token_id} fast={tok_L.is_fast}")
assert VOCAB_SIZE == M7_CFG["vocab_size"], f"tokenizer has {VOCAB_SIZE} entries but Model 7 was built for {M7_CFG['vocab_size']}"
assert None not in (tok_L.cls_token_id, tok_L.sep_token_id, tok_L.pad_token_id)

PAD_L, SEP_L, CLS_L = tok_L.pad_token_id, tok_L.sep_token_id, tok_L.cls_token_id
print("tokenizer lowercases:", tok_L.tokenize("Section 302 IPC, The Supreme Court"))

## Data

In [ ]:
def fetch(rel):
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=rel, repo_type="dataset", token=HF_TOKEN)

def read_json(rel):
    with open(fetch(rel), "r", encoding="utf-8") as fh:
        return json.load(fh)

def as_doc_list(data):
    """train/dev: list of {id, text, labels}; test: dict {doc_id: [sentences]}. Returns list of {id, text, labels?}."""
    if isinstance(data, dict):
        if "text" in data and isinstance(data["text"], list):
            data = [data]
        else:
            items = []
            for k, v in data.items():
                if isinstance(v, list):
                    items.append({"id": k, "text": v})
                elif isinstance(v, dict):
                    v = dict(v); v.setdefault("id", k)
                    items.append(v)
            data = items
    docs = []
    for d in data:
        did = str(d.get("id", d.get("doc_id", d.get("document_id"))))
        text = d.get("text", d.get("sentences"))
        docs.append({"id": did, "text": list(text), "labels": d.get("labels")})
    return docs

vocab_raw = read_json(f"{P3}/Train/label_vocab.json")
if isinstance(vocab_raw, dict) and all(isinstance(v, int) for v in vocab_raw.values()):
    LABELS = [k for k, _ in sorted(vocab_raw.items(), key=lambda kv: kv[1])]
elif isinstance(vocab_raw, dict) and all(isinstance(k, str) and k.isdigit() for k in vocab_raw):
    LABELS = [vocab_raw[k] for k in sorted(vocab_raw, key=int)]
elif isinstance(vocab_raw, dict) and "labels" in vocab_raw:
    LABELS = list(vocab_raw["labels"])
elif isinstance(vocab_raw, dict):
    LABELS = list(vocab_raw.keys())
else:
    LABELS = list(vocab_raw)
LABELS = [str(l) for l in LABELS]
L2I = {l: i for i, l in enumerate(LABELS)}
K = len(LABELS)
print(K, "labels:", LABELS)

splits = {}
for dom in ("CL", "IT"):
    for sp in ("train", "dev", "test"):
        folder = "Test" if sp == "test" else "Train"
        splits[(dom, sp)] = as_doc_list(read_json(f"{P3}/{folder}/{dom}_{sp}.json"))
        docs = splits[(dom, sp)]
        print(f"{dom} {sp:5s}: {len(docs):4d} docs, {sum(len(d['text']) for d in docs):6d} sentences")
        if sp != "test":
            for d in docs:
                assert d["labels"] is not None and len(d["labels"]) == len(d["text"]), d["id"]
                for l in d["labels"]:
                    assert l in L2I, f"unknown label {l!r}"
if SMOKE:
    for key in splits: splits[key] = splits[key][:3]
    A_EPOCHS = 1
cnt = Counter(l for dom in ("CL", "IT") for d in splits[(dom, "train")] for l in d["labels"])
print("train label counts:", {l: cnt[l] for l in LABELS})

def pos_bucket(i, n):
    return min(POS_BUCKETS - 1, int(POS_BUCKETS * i / max(n, 1)))

def head_text(dom, i, n):
    return (f"{dom} " if USE_DOMAIN_TAG else "") + f"position {pos_bucket(i, n)} of {POS_BUCKETS}."

## Shared training / evaluation helpers

In [ ]:
def macro_scores(y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=list(range(K)), average="macro", zero_division=0)
    return float(p), float(r), float(f)

def class_weights(y):
    freq = np.bincount(y, minlength=K).astype(float) + 1.0
    w = 1.0 / freq ** CLASS_WEIGHT_POWER
    w = w / (w * freq).sum() * freq.sum()          # average per-example weight stays 1
    return torch.tensor(w, dtype=torch.float32, device=dev)

def make_sched(opt, total_steps, warm_frac, floor):
    warm = max(1, int(warm_frac * total_steps))
    return torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else max(floor, (total_steps - s) / max(1, total_steps - warm)))

@torch.no_grad()
def predict_logits(model, collate_fn, ex, bs):
    model.eval()
    order = np.argsort([len(x) for x in ex["ids"]])
    out = np.zeros((len(order), K), dtype=np.float32)
    for s in range(0, len(order), bs):
        idx = order[s:s + bs]
        *inp, _y = collate_fn(ex, idx)
        with torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=use_amp):
            lg = model(*inp)
        out[idx] = lg.float().cpu().numpy()
    return out

def evaluate_dev(model, collate_fn, dev_ex, bs):
    res = {}
    for dom in ("CL", "IT"):
        res[dom] = macro_scores(dev_ex[dom]["y"], predict_logits(model, collate_fn, dev_ex[dom], bs).argmax(1))
    res["mean_f1"] = (res["CL"][2] + res["IT"][2]) / 2
    return res

def run_training(model, opt, sched, scaler, train_ex, dev_ex, collate_fn, ckpt_file, epochs, patience, batch, infer_bs, cls_w, tag, teacher_logits=None, kd_T=2.0, kd_alpha=0.5):
    """Early-stopped training; model selection on the mean of CL and IT dev macro-F1."""
    trainable = [p for p in model.parameters() if p.requires_grad]
    best, bad, t0 = -1.0, 0, time.time()
    N = len(train_ex["y"])
    for ep in range(1, epochs + 1):
        model.train()
        order = np.random.permutation(N); tot, nb = 0.0, 0
        bar = tqdm(range(0, N, batch), desc=f"{tag} ep{ep}/{epochs}", leave=False)
        for bi in bar:
            idx = order[bi:bi + batch]
            *inp, y = collate_fn(train_ex, idx)
            with torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=use_amp):
                lg = model(*inp)
            loss = F.cross_entropy(lg.float(), y, weight=cls_w)
            if teacher_logits is not None:                 # knowledge distillation: blend with the teacher's soft labels
                tl = torch.from_numpy(teacher_logits[idx]).to(dev)
                kd = F.kl_div(F.log_softmax(lg.float() / kd_T, -1), F.softmax(tl / kd_T, -1), reduction="batchmean") * kd_T * kd_T
                loss = (1.0 - kd_alpha) * loss + kd_alpha * kd
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            tot += float(loss.detach()); nb += 1
            bar.set_postfix(loss=f"{tot / nb:.4f}")
        r = evaluate_dev(model, collate_fn, dev_ex, infer_bs)
        print(f"  [{tag}] ep {ep}: loss {tot / nb:.4f} | CL P/R/F1 {r['CL'][0]:.3f}/{r['CL'][1]:.3f}/{r['CL'][2]:.3f} "
              f"| IT P/R/F1 {r['IT'][0]:.3f}/{r['IT'][1]:.3f}/{r['IT'][2]:.3f} | mean F1 {r['mean_f1']:.4f} | {(time.time() - t0) / 60:.1f} min")
        if r["mean_f1"] > best:
            best, bad = r["mean_f1"], 0
            torch.save({"model": model.state_dict(), "dev_f1": best, "epoch": ep}, ckpt_file)
        else:
            bad += 1
            if bad >= patience:
                print("  early stop"); break
    model.load_state_dict(torch.load(ckpt_file, map_location=dev)["model"])
    return best

def stage_outputs(model, collate_fn, dev_ex, test_ex, bs, tag):
    """Dev/test logits of a finished model + its stand-alone dev scores."""
    out = {"dev": {d: predict_logits(model, collate_fn, dev_ex[d], bs) for d in ("CL", "IT")},
           "test": {d: predict_logits(model, collate_fn, test_ex[d], bs) for d in ("CL", "IT")}}
    sc = {d: macro_scores(dev_ex[d]["y"], out["dev"][d].argmax(1)) for d in ("CL", "IT")}
    sc["mean_f1"] = (sc["CL"][2] + sc["IT"][2]) / 2
    out["score"] = sc
    print(f"[{tag}] stand-alone dev  CL P/R/F1 {sc['CL'][0]:.3f}/{sc['CL'][1]:.3f}/{sc['CL'][2]:.3f} | "
          f"IT {sc['IT'][0]:.3f}/{sc['IT'][1]:.3f}/{sc['IT'][2]:.3f} | mean F1 {sc['mean_f1']:.4f}")
    return out

## Stage A - Model 7 definition + checkpoint

In [ ]:
from transformers import BertConfig, BertModel


def build_model7():
    """Model 7 backbone: embeddings (token + learned position + segment, LayerNorm) -> 12 post-LN layers -> pooler (unused)."""
    return BertModel(BertConfig(**M7_CFG), add_pooling_layer=True)


def _norm_key(k):
    for pre in ("_orig_mod.", "module."):
        while k.startswith(pre):
            k = k[len(pre):]
    if CKPT_KEY_PREFIX and k.startswith(CKPT_KEY_PREFIX):
        k = k[len(CKPT_KEY_PREFIX):]
    return k.replace(".LayerNorm.gamma", ".LayerNorm.weight").replace(".LayerNorm.beta", ".LayerNorm.bias")


def split_checkpoint(raw):
    """Return (encoder_state_dict, prediction_head_state_dict) from a Model 7 checkpoint of any common wrapping."""
    sd = raw
    if isinstance(raw, dict):
        for k in ("model_state_dict", "model", "state_dict", "global_state_dict", "theta_g"):
            if k in raw and isinstance(raw[k], dict):
                sd = raw[k]; break
    enc, head = {}, {}
    for k, v in sd.items():
        if not torch.is_tensor(v):
            continue
        k = _norm_key(k)
        if k.startswith("bert."):
            enc[k[len("bert."):]] = v
        elif k.startswith("cls."):
            head[k] = v
        else:
            enc[k] = v
    return enc, head


class PredHead:
    """BERT prediction (LM) head used only for the load sanity check: dense -> GELU -> LayerNorm -> tied decoder (+ bias)."""
    def __init__(self, head_sd, emb_w):
        g = lambda *names: next((head_sd[n] for n in names if n in head_sd), None)
        self.dw = g("cls.predictions.transform.dense.weight"); self.db = g("cls.predictions.transform.dense.bias")
        self.lw = g("cls.predictions.transform.LayerNorm.weight"); self.lb = g("cls.predictions.transform.LayerNorm.bias")
        self.bias = g("cls.predictions.bias", "cls.predictions.decoder.bias")
        self.emb_w = emb_w
        self.ok = None not in (self.dw, self.db, self.lw, self.lb)

    def __call__(self, h):
        d = F.gelu(F.linear(h, self.dw.to(h.device, h.dtype), self.db.to(h.device, h.dtype)))
        d = F.layer_norm(d, (D_MODEL,), self.lw.to(h.device, h.dtype), self.lb.to(h.device, h.dtype), M7_CFG["layer_norm_eps"])
        b = self.bias.to(h.device, h.dtype) if self.bias is not None else None
        return F.linear(d, self.emb_w.to(h.dtype), b)

In [ ]:
base = build_model7()
ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
enc_sd, head_sd = split_checkpoint(ckpt)
print("checkpoint meta:", {k: ckpt[k] for k in ("step", "round", "loss", "val_loss") if isinstance(ckpt, dict) and k in ckpt})
missing, unexpected = base.load_state_dict(enc_sd, strict=False)
missing = [k for k in missing if not k.startswith("pooler.")]               # the pooler is never trained by the continuation loss
unexpected = [k for k in unexpected if k != "embeddings.position_ids"]
assert not missing, f"checkpoint mismatch - missing encoder keys: {missing[:6]} ... | unexpected: {unexpected[:6]}  (check CKPT_KEY_PREFIX)"
if unexpected:
    print("note: ignored unexpected keys:", unexpected[:6])
print(f"loaded {CKPT_PATH}: {len(enc_sd)} encoder tensors, {len(head_sd)} head tensors | encoder+pooler params {sum(p.numel() for p in base.parameters()) / 1e6:.2f}M")
del ckpt
base = base.to(dev).eval()

# sanity: with the prefix-LM mask Model 7 should predict the suffix of a legal text far better than chance (ln 30522 = 10.3).
ph = PredHead({k: v for k, v in head_sd.items()}, base.embeddings.word_embeddings.weight.detach())
if ph.ok:
    _txt = " ".join(splits[("CL", "train")][0]["text"])
    ids_ = tok_L(_txt, add_special_tokens=False)["input_ids"][:509]
    P = min(320, int(0.63 * len(ids_))); S = len(ids_) - P
    seq = [CLS_L] + ids_[:P] + [SEP_L] + ids_[P:] + [SEP_L]
    T, Lp = len(seq), P + 2
    idx = torch.arange(T, device=dev)
    pm = ((idx[None, :] < Lp) | (idx[None, :] <= idx[:, None])).long()[None]            # (1,T,T): prefix bidirectional, suffix causal
    tt = torch.tensor([[0] * Lp + [1] * (T - Lp)], device=dev)
    try:
        with torch.no_grad(), torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=use_amp):
            hs = base(input_ids=torch.tensor([seq], device=dev), attention_mask=pm, token_type_ids=tt).last_hidden_state
            lg = ph(hs[:, Lp - 1:T - 1])
        tgt = torch.tensor([seq[Lp:]], device=dev)
        lm_loss = F.cross_entropy(lg.float().reshape(-1, VOCAB_SIZE), tgt.reshape(-1)).item()
        print(f"continuation sanity loss on a training judgment ({S} suffix tokens): {lm_loss:.3f}  (chance = {math.log(VOCAB_SIZE):.2f}; a trained model is clearly lower)")
        assert lm_loss < 8.0, "continuation loss is near chance - weights or tokenizer do not match"
    except AssertionError:
        raise
    except Exception as e:
        print("sanity check skipped (3-D attention mask not supported by this transformers version):", repr(e)[:200])
else:
    print("prediction-head weights not found in the checkpoint - sanity check skipped (only the encoder is needed for classification)")
del head_sd, enc_sd, ph

## Stage A - sentence classifier on the Model 7 backbone

In [ ]:
class LoRALinear(nn.Module):
    """y = W x + (alpha/r) B A x   (base W frozen, B starts at zero)."""
    def __init__(self, base_linear, rank, alpha):
        super().__init__()
        self.base = base_linear
        out_f, in_f = base_linear.weight.shape
        self.A = nn.Parameter(torch.randn(rank, in_f) * 0.02)
        self.B = nn.Parameter(torch.zeros(out_f, rank))
        self.scale = alpha / rank

    def forward(self, x):
        return self.base(x) + self.scale * ((x @ self.A.t()) @ self.B.t())


class LoRMALinear(nn.Module):
    """LoRMA (multiplicative): y = (I + (alpha/r) B A) W x   (base W frozen, B starts at zero -> identical to Model 7 at step 0)."""
    def __init__(self, base_linear, rank, alpha):
        super().__init__()
        self.base = base_linear
        out_f = base_linear.weight.shape[0]
        self.A = nn.Parameter(torch.randn(rank, out_f) * 0.02)
        self.B = nn.Parameter(torch.zeros(out_f, rank))
        self.scale = alpha / rank

    def forward(self, x):
        h = self.base(x)
        return h + self.scale * ((h @ self.A.t()) @ self.B.t())


def adapt_linears(backbone, kind, rank, alpha):
    """Wrap every nn.Linear inside the 12 Transformer layers: Q, K, V, attention output, FFN up and FFN down (6 per layer)."""
    wrapper = LoRMALinear if kind == "lorma" else LoRALinear
    n = 0
    for blk in backbone.encoder.layer:
        targets = [(parent, name, child) for parent in blk.modules() for name, child in parent.named_children() if isinstance(child, nn.Linear)]
        for parent, name, child in targets:
            setattr(parent, name, wrapper(child, rank, alpha))
            n += 1
    return n

In [ ]:
class Model7RR(nn.Module):
    """Model 7 backbone used as a bidirectional encoder (whole input = prefix, segment 0); head on [mean of target-sentence tokens | mean of all tokens]."""
    def __init__(self, backbone, n_labels, use_gc):
        super().__init__()
        self.backbone = backbone
        if use_gc:
            backbone.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        self.drop = nn.Dropout(0.1)
        self.head = nn.Linear(2 * D_MODEL, n_labels)
        nn.init.normal_(self.head.weight, std=0.02); nn.init.zeros_(self.head.bias)

    def forward(self, ids, pad_mask, tmask):
        h = self.backbone(input_ids=ids, attention_mask=pad_mask.long(), token_type_ids=torch.zeros_like(ids)).last_hidden_state.float()
        tm = tmask.float().unsqueeze(-1); am = pad_mask.float().unsqueeze(-1)
        mean_t = (h * tm).sum(1) / tm.sum(1).clamp(min=1)
        mean_a = (h * am).sum(1) / am.sum(1).clamp(min=1)
        return self.head(self.drop(torch.cat([mean_t, mean_a], dim=-1)))

# ---- inputs for Model A:  [CLS] head [SEP] previous: ctx [SEP] ctx ... [SEP] target sentence: TARGET [SEP] next: ctx [SEP]
def _enc(texts):
    return tok_L(texts, add_special_tokens=False)["input_ids"]

M_PREV, M_TGT, M_NEXT = _enc(["previous:", "target sentence:", "next:"])
_head_cache = {}

def build_examples_A(docs, dom, with_labels=True):
    ids_l, tm_l, y_l, meta = [], [], [], []
    for d in docs:
        sents, n = d["text"], len(d["text"])
        sid = _enc(sents)
        for i in range(n):
            hk = (dom, pos_bucket(i, n))
            if hk not in _head_cache:
                _head_cache[hk] = _enc([head_text(dom, i, n)])[0]
            prev = []
            for x in sid[max(0, i - PREV_K):i]:
                prev += x[:A_CTX_TOKENS] + [SEP_L]
            nxt = []
            for x in sid[i + 1:i + 1 + NEXT_K]:
                nxt += x[:A_CTX_TOKENS] + [SEP_L]
            tgt = sid[i][:A_TGT_TOKENS]
            body = _head_cache[hk] + [SEP_L] + M_PREV + prev + M_TGT      # everything before the target, after [CLS]
            post = [SEP_L] + M_NEXT + nxt
            budget = A_MAX_LEN - len(tgt) - 1                             # 1 position is kept for [CLS]
            if len(body) + len(post) > budget:                            # drop the oldest context / the tail of "next"
                pb = min(len(post), budget // 3)
                post = post[:pb]
                body = body[-(budget - pb):] if budget - pb > 0 else []
            pre = [CLS_L] + body
            seq = pre + tgt + post
            tm = [0] * len(pre) + [1] * len(tgt) + [0] * len(post)
            ids_l.append(seq); tm_l.append(tm)
            y_l.append(L2I[d["labels"][i]] if with_labels else -1)
            meta.append((d["id"], i))
    return {"ids": ids_l, "tm": tm_l, "y": y_l, "meta": meta}

def collate_A(ex, idx):
    L = max(len(ex["ids"][i]) for i in idx)
    ids = torch.full((len(idx), L), PAD_L, dtype=torch.long)
    pm = torch.zeros((len(idx), L), dtype=torch.bool)
    tm = torch.zeros((len(idx), L), dtype=torch.bool)
    for r, i in enumerate(idx):
        n = len(ex["ids"][i])
        ids[r, :n] = torch.tensor(ex["ids"][i]); pm[r, :n] = True; tm[r, :n] = torch.tensor(ex["tm"][i], dtype=torch.bool)
    y = torch.tensor([ex["y"][i] for i in idx], dtype=torch.long)
    return ids.to(dev), pm.to(dev), tm.to(dev), y.to(dev)

def cat_ex(parts):
    out = {k: [] for k in parts[0]}
    for p in parts:
        for k in out: out[k] += p[k]
    return out

trainA = cat_ex([build_examples_A(splits[(dom, "train")], dom) for dom in ("CL", "IT")])
devA = {dom: build_examples_A(splits[(dom, "dev")], dom) for dom in ("CL", "IT")}
testA = {dom: build_examples_A(splits[(dom, "test")], dom, with_labels=False) for dom in ("CL", "IT")}
lens = np.array([len(x) for x in trainA["ids"]])
print(f"[A] train {len(trainA['y'])} | dev { {d: len(v['y']) for d, v in devA.items()} } | test { {d: len(v['ids']) for d, v in testA.items()} }")
print(f"[A] token length: mean {lens.mean():.0f}, p95 {np.percentile(lens, 95):.0f}, max {lens.max()} (cap {A_MAX_LEN})")
assert lens.max() <= A_MAX_LEN <= MAX_SEQ_LEN
print("[A] example input:", tok_L.decode(trainA["ids"][5])[:600].replace("\n", " "))

## Stage A - adapt (full / LoRA / LoRMA) and train

In [ ]:
def build_optim(model, n_train, lr):
    bb = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith("backbone.")]
    hd = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("backbone.")]
    opt = torch.optim.AdamW([{"params": hd, "lr": A_HEAD_LR}, {"params": bb, "lr": lr}], weight_decay=A_WD)
    steps = math.ceil(n_train / A_BATCH) * A_EPOCHS
    return opt, make_sched(opt, steps, A_WARMUP, 0.05), torch.amp.GradScaler("cuda", enabled=use_amp)

def n_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad), sum(p.numel() for p in m.parameters())

def unfreeze_all(m):
    for p in m.parameters():
        p.requires_grad_(True)
    for p in m.pooler.parameters():             # unused by the classifier
        p.requires_grad_(False)

def make_wrapper(backbone, gc_flag):
    return Model7RR(backbone, K, gc_flag)

def make_student(pre_sd, layer_map):
    """Half-depth BERT: embeddings / pooler copied, layer i <- pre-trained layer layer_map[i]."""
    st = BertModel(BertConfig(**{**M7_CFG, "num_hidden_layers": len(layer_map)}), add_pooling_layer=True)
    new = {k: v for k, v in pre_sd.items() if not k.startswith("encoder.layer.")}
    for i, src in enumerate(layer_map):
        pref = f"encoder.layer.{src}."
        for k, v in pre_sd.items():
            if k.startswith(pref):
                new[f"encoder.layer.{i}." + k[len(pref):]] = v
    missing, unexpected = st.load_state_dict(new, strict=False)
    assert not missing, missing[:4]
    for p in st.pooler.parameters():
        p.requires_grad_(False)                             # unused by the classifier
    return st

cwA = class_weights(trainA["y"])
print("A class weights:", {l: round(float(x), 2) for l, x in zip(LABELS, cwA.cpu().numpy())})
lora_file = os.path.join(OUT_DIR, f"best_{METHOD}_rr.pt")
SKIP_TRAIN = bool((not RETRAIN_LORA) and LORA_CKPT and os.path.exists(LORA_CKPT))
TEACHER_SCORE, T_TRAIN = None, None

if METHOD == "kd":
    # ---------- stage 1: TEACHER = the full-depth model, every weight fine-tuned ----------
    PRE_SD = {k: v.detach().cpu().clone() for k, v in base.state_dict().items()}   # pristine pre-trained weights for the student
    if not SKIP_TRAIN:
        unfreeze_all(base)
        teacher = make_wrapper(base, _PRESETS["full"]["gc"]).to(dev)
        t_tr, t_tot = n_params(teacher)
        print(f"[teacher] full fine-tuning, {t_tr / 1e6:.2f}M trainable parameters")
        teacher_file = os.path.join(OUT_DIR, "best_teacher_rr.pt")
        if KD_TEACHER_CKPT and os.path.exists(KD_TEACHER_CKPT):
            teacher.load_state_dict(torch.load(KD_TEACHER_CKPT, map_location=dev)["model"])
            print("teacher loaded from", KD_TEACHER_CKPT)
        else:
            t_opt, t_sched, t_scaler = build_optim(teacher, len(trainA["y"]), KD_TEACHER_LR)
            run_training(teacher, t_opt, t_sched, t_scaler, trainA, devA, collate_A, teacher_file, A_EPOCHS, A_PATIENCE,
                         A_BATCH, A_INFER_BATCH, cwA, "teacher")
            del t_opt, t_sched, t_scaler
        TEACHER_SCORE = evaluate_dev(teacher, collate_A, devA, A_INFER_BATCH)
        print(f"[teacher] dev  CL F1 {TEACHER_SCORE['CL'][2]:.3f} | IT F1 {TEACHER_SCORE['IT'][2]:.3f} | mean F1 {TEACHER_SCORE['mean_f1']:.4f}")
        T_TRAIN = predict_logits(teacher, collate_A, trainA, A_INFER_BATCH)          # soft labels for every training sentence
        del teacher; base = None
        gc.collect(); torch.cuda.empty_cache()
    # ---------- stage 2: STUDENT = half the layers, initialised from evenly spaced pre-trained layers ----------
    base = make_student(PRE_SD, KD_LAYER_MAP).to(dev)
    del PRE_SD
    print(f"[student] {len(KD_LAYER_MAP)} layers copied from pre-trained layers {KD_LAYER_MAP}; all weights trainable")
elif METHOD == "full":
    unfreeze_all(base)
    print("full fine-tuning: all Phase-1 weights are trainable")
else:
    for p in base.parameters():
        p.requires_grad_(False)
    print(f"wrapped {adapt_linears(base, METHOD, RANK, ALPHA)} linear layers with {METHOD}")

modelA = make_wrapper(base, USE_GC).to(dev)
tr, tot = n_params(modelA)
print(f"[A] parameters: trainable {tr / 1e6:.2f}M / total {tot / 1e6:.2f}M ({tr / tot:.1%})")
optA, schedA, scalerA = build_optim(modelA, len(trainA["y"]), A_LR)

if SKIP_TRAIN:
    modelA.load_state_dict(torch.load(LORA_CKPT, map_location=dev)["model"])
    print(f"{METHOD} model loaded from", LORA_CKPT)
else:
    kd_kw = dict(teacher_logits=T_TRAIN, kd_T=KD_TEMP, kd_alpha=KD_ALPHA) if METHOD == "kd" else {}
    run_training(modelA, optA, schedA, scalerA, trainA, devA, collate_A, lora_file, A_EPOCHS, A_PATIENCE, A_BATCH, A_INFER_BATCH, cwA, METHOD, **kd_kw)
OUT_A = stage_outputs(modelA, collate_A, devA, testA, A_INFER_BATCH, METHOD)
META_TEST = {d: testA[d]["meta"] for d in ("CL", "IT")}
META_DEV = {d: devA[d]["meta"] for d in ("CL", "IT")}
Y_DEV = {d: np.array(devA[d]["y"]) for d in ("CL", "IT")}
del modelA, optA, schedA, scalerA, base, trainA, devA, testA
gc.collect(); torch.cuda.empty_cache()


## Dev report

In [ ]:
Y_DEV_PRED = {d: OUT_A["dev"][d].argmax(1) for d in ("CL", "IT")}
final = {d: macro_scores(Y_DEV[d], Y_DEV_PRED[d]) for d in ("CL", "IT")}
final["mean_f1"] = (final["CL"][2] + final["IT"][2]) / 2
print(f"Model 7 ({METHOD}) dev macro-F1:  CL P/R/F1 {final['CL'][0]:.3f}/{final['CL'][1]:.3f}/{final['CL'][2]:.3f} | "
      f"IT {final['IT'][0]:.3f}/{final['IT'][1]:.3f}/{final['IT'][2]:.3f} | mean F1 {final['mean_f1']:.4f}")
print("note: early stopping used this dev set, so the dev score is slightly optimistic.")
if TEACHER_SCORE is not None:
    print(f"KD: teacher (full fine-tune) dev mean F1 {TEACHER_SCORE['mean_f1']:.4f} -> distilled student {final['mean_f1']:.4f}")
for d in ("CL", "IT"):
    print(f"\n===== {d} dev =====")
    print(classification_report(Y_DEV[d], Y_DEV_PRED[d], labels=list(range(K)), target_names=LABELS, zero_division=0, digits=3))

np.savez_compressed(os.path.join(OUT_DIR, "model7_logits.npz"),
                    **{f"{s}_{d}": OUT_A[s][d] for s in ("dev", "test") for d in ("CL", "IT")},
                    **{f"ydev_{d}": Y_DEV[d] for d in ("CL", "IT")})

## Predict test and write submission

In [ ]:
predictions = {}
for d in ("CL", "IT"):
    pred = OUT_A["test"][d].argmax(1)
    per_doc = {}
    for (did, i), p in zip(META_TEST[d], pred):
        per_doc.setdefault(did, {})[i] = LABELS[int(p)]
    for doc in splits[(d, "test")]:
        labs = [per_doc[doc["id"]][i] for i in range(len(doc["text"]))]
        assert len(labs) == len(doc["text"])
        assert doc["id"] not in predictions, f"doc id {doc['id']} appears in both CL and IT test"
        predictions[doc["id"]] = labs
assert all(l in L2I for v in predictions.values() for l in v)

pred_path = os.path.join(OUT_DIR, "prediction.json")
with open(pred_path, "w", encoding="utf-8") as fh:
    json.dump(predictions, fh, ensure_ascii=False)
zip_path = f"/kaggle/working/submission_rr_model7_{METHOD}.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(pred_path, arcname="prediction.json")

with open(os.path.join(OUT_DIR, "model7_report.json"), "w") as fh:
    json.dump({"model": "model7", "method": METHOD, "dev_mean_f1": final["mean_f1"], "teacher_dev": TEACHER_SCORE,
               "dev_CL": {"precision": final["CL"][0], "recall": final["CL"][1], "f1": final["CL"][2]},
               "dev_IT": {"precision": final["IT"][0], "recall": final["IT"][1], "f1": final["IT"][2]}}, fh, indent=2)
print(f"wrote {zip_path}: {len(predictions)} docs, {sum(len(v) for v in predictions.values())} sentences")
print("test label distribution:", Counter(l for v in predictions.values() for l in v).most_common())